# 23 - Triton Ternary Matmul Spike

Spike técnico para validar um kernel Triton de matmul ternária/packed antes de integrar no runtime completo.

O que este notebook responde:
- conseguimos calcular `x @ W.T` lendo o `mlp_ternary_2bit.bin` sem reconstruir o peso inteiro persistente?
- o erro numérico bate contra `F.linear` com peso reconstruído?
- a latência é melhor que o protótipo PyTorch que desempacota no `forward`?

O que este notebook ainda não prova:
- não integra no modelo completo;
- não substitui o notebook 22;
- não fecha robustez/produto;
- é um spike de kernel para decidir se vale avançar para o notebook 24.

In [ ]:
# Dependências. Rode esta célula primeiro no Colab/T4.
import sys, subprocess, importlib.util

required = {
    'torch': 'torch',
    'triton': 'triton',
    'numpy': 'numpy',
    'psutil': 'psutil',
}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    print('Instalando dependências ausentes:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
else:
    print('Dependências principais já disponíveis.')

In [ ]:
# Bootstrap Colab/local.
from pathlib import Path
import os, sys, json, time, hashlib, gc, platform

IN_COLAB = 'google.colab' in sys.modules
PROJECT_NAME = 'gemma4_theory'
COLAB_DRIVE_ROOT = '/content/drive/MyDrive/RPT'
LOCAL_ROOT = str(Path.cwd().resolve())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path(COLAB_DRIVE_ROOT) / PROJECT_NAME
else:
    PROJECT_ROOT = Path(LOCAL_ROOT) / PROJECT_NAME if Path(LOCAL_ROOT).name != PROJECT_NAME else Path(LOCAL_ROOT)

EXPORTS_DIR = PROJECT_ROOT / 'exports'
DEPLOY_DIR = PROJECT_ROOT / 'deploy'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'packed_recovery'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PACKAGE_DIR = next((p for p in [EXPORTS_DIR / 'ternary_mlp_l78', DEPLOY_DIR] if (p / 'manifest.json').exists() and (p / 'mlp_ternary_2bit.bin').exists()), None)
if PACKAGE_DIR is None:
    raise FileNotFoundError('Packed package not found in exports/ternary_mlp_l78 or deploy')

MANIFEST_PATH = PACKAGE_DIR / 'manifest.json'
BIN_PATH = PACKAGE_DIR / 'mlp_ternary_2bit.bin'
EXPECTED_PACKED_SHA256 = '0b6849fe3ceaab615a4e54f63f5a96e9d9d431ba03b7b791a7ab0638ea174116'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('PACKAGE_DIR =', PACKAGE_DIR)
print('RESULTS_DIR =', RESULTS_DIR)

In [ ]:
# Utilitários de leitura/reconstrução e seleção de tensores reais.
import numpy as np
import torch
import torch.nn.functional as F
import triton
import triton.language as tl

assert torch.cuda.is_available(), 'Este spike precisa de GPU CUDA para Triton.'

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

def load_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def unpack_codes_from_blob(blob, n):
    raw = np.frombuffer(blob, dtype=np.uint8)
    codes = np.empty(raw.size * 4, dtype=np.uint8)
    codes[0::4] = raw & 0b11
    codes[1::4] = (raw >> 2) & 0b11
    codes[2::4] = (raw >> 4) & 0b11
    codes[3::4] = (raw >> 6) & 0b11
    return codes[:n]

def unpack_tensor_from_blob(blob, n, shape, scale, dtype=torch.float16, device='cuda'):
    codes = unpack_codes_from_blob(blob, n)
    values = np.zeros(n, dtype=np.float32)
    values[codes == 1] = scale
    values[codes == 2] = -scale
    return torch.from_numpy(values).view(tuple(shape)).to(device=device, dtype=dtype)

def read_tensor_blob(item):
    with open(BIN_PATH, 'rb') as handle:
        handle.seek(item['offset'])
        return handle.read(item['packed_bytes'])

manifest = load_json(MANIFEST_PATH)
packed_sha = sha256_file(BIN_PATH)
assert packed_sha == EXPECTED_PACKED_SHA256, packed_sha
assert manifest['summary']['num_tensors'] == 78

gate_item = next(item for item in manifest['tensors'] if item['name'] == 'model.layers.0.mlp.gate_proj.weight')
down_item = next(item for item in manifest['tensors'] if item['name'] == 'model.layers.0.mlp.down_proj.weight')
print('gate shape:', gate_item['shape'])
print('down shape:', down_item['shape'])
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Kernel Triton: cada programa calcula um tile [BLOCK_M, BLOCK_N] de C = X @ W.T.
# W está em 2-bit codes: 0 -> 0, 1 -> +scale, 2 -> -scale, 3 reservado/tratado como 0.
@triton.jit
def ternary_linear_kernel(
    x_ptr,
    packed_ptr,
    out_ptr,
    M: tl.constexpr,
    K: tl.constexpr,
    N: tl.constexpr,
    scale: tl.constexpr,
    stride_xm: tl.constexpr,
    stride_xk: tl.constexpr,
    stride_om: tl.constexpr,
    stride_on: tl.constexpr,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

    for k0 in range(0, K, BLOCK_K):
        k_idxs = k0 + offs_k
        x = tl.load(x_ptr + offs_m[:, None] * stride_xm + k_idxs[None, :] * stride_xk, mask=(offs_m[:, None] < M) & (k_idxs[None, :] < K), other=0.0)

        linear = offs_n[:, None] * K + k_idxs[None, :]
        byte_idx = linear // 4
        shift = (linear % 4) * 2
        raw = tl.load(packed_ptr + byte_idx, mask=(offs_n[:, None] < N) & (k_idxs[None, :] < K), other=0).to(tl.uint32)
        code = (raw >> shift) & 3
        w_pos = code == 1
        w_neg = code == 2
        w = tl.where(w_pos, scale, tl.where(w_neg, -scale, 0.0)).to(x.dtype)
        acc += tl.dot(x, tl.trans(w), out_dtype=tl.float32)

    tl.store(out_ptr + offs_m[:, None] * stride_om + offs_n[None, :] * stride_on, acc, mask=(offs_m[:, None] < M) & (offs_n[None, :] < N))

def triton_ternary_linear(x, packed_u8, item, *, block_m=16, block_n=16, block_k=64):
    shape = item['shape']
    n_out, k_in = int(shape[0]), int(shape[1])
    assert x.shape[-1] == k_in, (x.shape, shape)
    x2 = x.reshape(-1, k_in).contiguous()
    out = torch.empty((x2.shape[0], n_out), device=x.device, dtype=torch.float32)
    grid = (triton.cdiv(x2.shape[0], block_m), triton.cdiv(n_out, block_n))
    ternary_linear_kernel[grid](
        x2,
        packed_u8,
        out,
        x2.shape[0],
        k_in,
        n_out,
        float(item['scale']),
        x2.stride(0),
        x2.stride(1),
        out.stride(0),
        out.stride(1),
        BLOCK_M=block_m,
        BLOCK_N=block_n,
        BLOCK_K=block_k,
    )
    return out.reshape(*x.shape[:-1], n_out)

In [ ]:
# Bench helpers.
def cuda_sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def bench(fn, *, warmup=5, iters=20):
    for _ in range(warmup):
        y = fn()
    cuda_sync()
    start = time.perf_counter()
    for _ in range(iters):
        y = fn()
    cuda_sync()
    elapsed = time.perf_counter() - start
    return {'iters': iters, 'total_sec': elapsed, 'avg_ms': elapsed * 1000 / iters}

def make_case(item, m_tokens=1, dtype=torch.float16):
    blob = read_tensor_blob(item)
    packed = torch.from_numpy(np.frombuffer(blob, dtype=np.uint8).copy()).to('cuda')
    weight = unpack_tensor_from_blob(blob, item['numel'], item['shape'], item['scale'], dtype=dtype, device='cuda')
    x = torch.randn((m_tokens, int(item['shape'][1])), device='cuda', dtype=dtype)
    return x, packed, weight

def run_case(label, item, m_tokens=1, dtype=torch.float16, block_m=16, block_n=16, block_k=64):
    torch.cuda.empty_cache()
    x, packed, weight = make_case(item, m_tokens=m_tokens, dtype=dtype)
    ref = F.linear(x, weight).float()
    tri = triton_ternary_linear(x, packed, item, block_m=block_m, block_n=block_n, block_k=block_k).float()
    cuda_sync()
    max_abs_err = float((ref - tri).abs().max().item())
    mean_abs_err = float((ref - tri).abs().mean().item())
    ref_norm = float(ref.abs().mean().item())

    dense_bench = bench(lambda: F.linear(x, weight), warmup=5, iters=30)
    triton_bench = bench(lambda: triton_ternary_linear(x, packed, item, block_m=block_m, block_n=block_n, block_k=block_k), warmup=5, iters=30)

    # Baseline mais próximo do runtime 22: desempacotar peso e fazer linear a cada forward.
    blob = read_tensor_blob(item)
    def unpack_then_linear():
        w = unpack_tensor_from_blob(blob, item['numel'], item['shape'], item['scale'], dtype=dtype, device='cuda')
        return F.linear(x, w)
    unpack_bench = bench(unpack_then_linear, warmup=2, iters=10)

    row = {
        'label': label,
        'tensor_name': item['name'],
        'shape': item['shape'],
        'm_tokens': m_tokens,
        'dtype': str(dtype),
        'block_m': block_m,
        'block_n': block_n,
        'block_k': block_k,
        'max_abs_err': max_abs_err,
        'mean_abs_err': mean_abs_err,
        'ref_abs_mean': ref_norm,
        'dense_bench': dense_bench,
        'unpack_then_linear_bench': unpack_bench,
        'triton_bench': triton_bench,
        'speedup_vs_unpack': unpack_bench['avg_ms'] / triton_bench['avg_ms'] if triton_bench['avg_ms'] > 0 else None,
        'speedup_vs_dense': dense_bench['avg_ms'] / triton_bench['avg_ms'] if triton_bench['avg_ms'] > 0 else None,
    }
    print(json.dumps(row, ensure_ascii=False, indent=2))
    del x, packed, weight, ref, tri
    torch.cuda.empty_cache()
    return row

In [ ]:
# Executa os casos principais. M=1 simula geração token-a-token; M=16 simula batch/seq pequeno.
cases = []
for m_tokens in [1, 16]:
    cases.append(run_case('gate_proj_real_shape', gate_item, m_tokens=m_tokens, block_m=16, block_n=16, block_k=64))
    cases.append(run_case('down_proj_real_shape', down_item, m_tokens=m_tokens, block_m=16, block_n=16, block_k=64))

success = all(row['max_abs_err'] <= 2e-2 for row in cases)
summary = {
    'success': success,
    'num_cases': len(cases),
    'max_abs_err_max': max(row['max_abs_err'] for row in cases),
    'mean_speedup_vs_unpack': sum(row['speedup_vs_unpack'] for row in cases) / len(cases),
    'mean_speedup_vs_dense': sum(row['speedup_vs_dense'] for row in cases) / len(cases),
}
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
# Salva relatório.
report = {
    'notebook': '23_triton_ternary_matmul_spike',
    'purpose': 'Triton packed ternary matmul spike before full runtime integration.',
    'created_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    'environment': {
        'platform': platform.platform(),
        'python': sys.version,
        'torch': torch.__version__,
        'triton': triton.__version__,
        'cuda_device': torch.cuda.get_device_name(0),
    },
    'packed_sha256': packed_sha,
    'summary': summary,
    'cases': cases,
    'baseline_context': {
        'notebook_22_native_structural_accuracy': 1.0,
        'notebook_22_native_structural_tokens_per_sec': 2.285913541069469,
        'notebook_20_materialized_tokens_per_sec': 7.746480522791005,
    },
    'next_step': 'If success is true and speedup_vs_unpack is meaningful, integrate kernel into PackedTernaryLinear in notebook 24.',
}
out_path = RESULTS_DIR / f"23_triton_ternary_matmul_spike_{time.strftime('%Y%m%d_%H%M%S')}.json"
out_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('saved:', out_path)

## Critério de avanço para o 24

Avançar para integração no runtime completo somente se:

1. `success == true`;
2. erro máximo contra `F.linear` ficar aceitável para fp16;
3. `speedup_vs_unpack` for maior que `1.0` nos shapes reais;
4. a implementação não aumentar memória persistente.

Se passar, o `24_integrate_triton_ternary_runtime.ipynb` deve trocar o `PackedTernaryLinear.forward` para usar o kernel Triton e repetir a avaliação `100/100` do notebook 22.